# Adding, Modifying & Removing Columns/Rows

## Definition
After loading data, you constantly need to **add new features**, **fix existing columns**, and **remove unnecessary data**.
This is the core of feature engineering.

### Operations
| Operation | Method |
|-----------|--------|
| Add column | `df['new_col'] = values` |
| Add column (safe) | `df.assign(new_col=values)` |
| Modify column | `df['col'] = df['col'].apply(...)` |
| Rename column | `df.rename(columns={'old': 'new'})` |
| Remove column | `df.drop(columns=['col'])` |
| Add row | `pd.concat([df, new_row])` |
| Remove rows | `df.drop(index=[...])` |
| Remove by condition | `df[~condition]` |
| Insert at position | `df.insert(pos, name, values)` |

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'salary': [75000, 85000, 65000, 95000, 70000],
    'yoe':    [3, 8, 1, 18, 4],
    'dept':   ['Eng', 'Mkt', 'Eng', 'HR', 'Eng']
})
print(df)

In [ ]:
# Adding new columns
df['tax'] = df['salary'] * 0.25
df['net_salary'] = df['salary'] - df['tax']
df['senior'] = df['yoe'] >= 5
print(df)

In [ ]:
# df.assign() — returns new DataFrame, great for chaining
df2 = (df
    .assign(bonus=lambda x: x['salary'] * 0.1)
    .assign(total_comp=lambda x: x['salary'] + x['bonus'])
)
print(df2[['name', 'salary', 'bonus', 'total_comp']])

In [ ]:
# Modifying existing columns
df['salary'] = df['salary'].round(-3)  # round to nearest 1000
df['name'] = df['name'].str.upper()    # uppercase names
print(df[['name', 'salary']])

In [ ]:
# Renaming columns
df_renamed = df.rename(columns={
    'name': 'employee_name',
    'yoe': 'years_of_experience'
})
print(df_renamed.columns.tolist())

# Rename all columns at once
df.columns = [c.lower().replace(' ', '_') for c in df.columns]
print('After bulk rename:', df.columns.tolist())

In [ ]:
# Removing columns
df_dropped = df.drop(columns=['tax', 'net_salary'])
print('After dropping columns:')
print(df_dropped.columns.tolist())

# Remove multiple columns
cols_to_drop = ['senior', 'bonus'] if 'bonus' in df.columns else ['senior']
df2 = df.drop(columns=cols_to_drop, errors='ignore')  # errors='ignore' avoids KeyError

In [ ]:
# Inserting column at specific position
df3 = df.copy()
df3.insert(0, 'employee_id', range(1, len(df3)+1))
print(df3)

In [ ]:
# Adding rows
new_rows = pd.DataFrame([
    {'name': 'FRANK', 'salary': 90000, 'yoe': 12, 'dept': 'Mkt', 'tax': 22500, 'net_salary': 67500, 'senior': True},
])
df_extended = pd.concat([df, new_rows], ignore_index=True)
print(f'After adding row: {df_extended.shape}')

In [ ]:
# Removing rows
# By index position
df_no_first = df.drop(index=0)
print('After dropping row 0:')
print(df_no_first)

# By condition
df_no_hr = df[df['dept'] != 'HR']
print('\nAfter removing HR dept:')
print(df_no_hr)

In [ ]:
# EXCEPTION: Setting on a copy (SettingWithCopyWarning)
import warnings

# WRONG — may not work as expected
sub = df[df['dept'] == 'Eng']
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    sub['salary'] = 99999
    if w:
        print('Warning:', w[0].category.__name__)

# CORRECT — modify original with .loc
df.loc[df['dept'] == 'Eng', 'salary'] = 99999
print('\nCorrect modification:')
print(df[['name', 'dept', 'salary']])